In [ ]:
import os
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


@dataclass
class Config:
    seed: int = 42
    batch_size: int = 128
    epochs: int = 5
    lr: float = 1e-3
    weight_decay: float = 0.0
    num_workers: int = 2
    val_split: int = 10_000
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = Config()
set_seed(cfg.seed)
print("Device:", cfg.device)



# Transformaciones:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

data_dir = "./data"

train_full = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)
test_ds    = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)

#  train = train + val
val_size = cfg.val_split
train_size = len(train_full) - val_size
train_ds, val_ds = random_split(
    train_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(cfg.seed)
)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=(cfg.device=="cuda"))
val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=(cfg.device=="cuda"))
test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=(cfg.device=="cuda"))

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")



class MNISTSequentialCNN(nn.Module):
    """
    CNN sencilla y efectiva para MNIST.
    Estructura:
      - Conv -> ReLU -> Conv -> ReLU -> MaxPool
      - Conv -> ReLU -> Conv -> ReLU -> MaxPool
      - Flatten -> Linear -> ReLU -> Dropout -> Linear (10 clases)
    """
    def __init__(self, num_classes: int = 10, dropout: float = 0.2):
        super().__init__()

        # Bloque convolucional (secuencial)
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 1x28x28 -> 32x28x28
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), # 32x28x28 -> 32x28x28
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                             # 32x28x28 -> 32x14x14

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 32x14x14 -> 64x14x14
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), # 64x14x14 -> 64x14x14
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                             # 64x14x14 -> 64x7x7
        )

        # Cabeza (clasificador) (secuencial)
        self.classifier = nn.Sequential(
            nn.Flatten(),                 # 64x7x7 -> 3136
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x


model = MNISTSequentialCNN().to(cfg.device)
print(model)


def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: str):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = F.cross_entropy(logits, y)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, device: str):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches



optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)


###train
best_val_acc = -1.0
best_path = "best_mnist_model.pt"

for epoch in range(1, cfg.epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, cfg.device)
    val_loss, val_acc = evaluate(model, val_loader, cfg.device)


    print(
        f"Epoch {epoch:02d}/{cfg.epochs} | "
        f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f}"
    )

    # Guardar mejor modelo por val_acc
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": cfg.__dict__,
            "val_acc": best_val_acc
        }, best_path)

print("Mejor val_acc:", best_val_acc)
print("Guardado en:", best_path)



ckpt = torch.load(best_path, map_location=cfg.device)
model.load_state_dict(ckpt["model_state_dict"])

test_loss, test_acc = evaluate(model, test_loader, cfg.device)
print(f"TEST | loss {test_loss:.4f} acc {test_acc:.4f}")


Device: cpu


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.09MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.9MB/s]

Train: 50000 | Val: 10000 | Test: 10000
MNISTSequentialCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


Epoch 01/5 | train loss 0.2193 acc 0.9294 | val loss 0.0722 acc 0.9776
Epoch 02/5 | train loss 0.0586 acc 0.9824 | val loss 0.0402 acc 0.9884
Epoch 03/5 | train loss 0.0397 acc 0.9876 | val loss 0.0366 acc 0.9886
Epoch 04/5 | train loss 0.0315 acc 0.9903 | val loss 0.0330 acc 0.9900
Epoch 05/5 | train loss 0.0221 acc 0.9929 | val loss 0.0410 acc 0.9891
Mejor val_acc: 0.9900118670886076
Guardado en: best_mnist_model.pt
TEST | loss 0.0250 acc 0.9918
